End-to-End FT Worfklows
-----------
In this tutorial, we describe the full end-to-end workflow to take a circuit written in any gate set to one written in a suitable gate set for many QECs (Quantum Error Correcting codes), the Clifford + T gate set.

This tutorial requires the `bqskit` and `bqskit-ft` extension package and can be install via pip:
```bash
pip install bqskit bqskit-ft
```

Contents
-----------
* [Continuous Gate Set Re-Targeting](#continuous-gate-set-retargeting)
* [Decomposing to a Dicrete Gate Set](#fault-tolerant-gate-sets)
    * Multi-Qubit Gate Synthesis 
    * Single qubit U3 Gate Synthesis
    * ZXZXZ Decomposition with Gridsynth 
* [Algorithmic Error](#algorithmic-error)
* [Pauli Product Measurements](#compiling-to-ppms)

Continuous Gate Set Re-Targeting
----------
One of the biggest advantages of the BQSKit workflow is our portability. Since we define our gates numerically, we can accept any gate type as long as it can be correctly represented by an underlying unitary. 

We can start by defining a circuit with various gates and unitaries. In our example we start by defining a circuit with random U3 gates, CNOT Gates, as well as some unitary operators corresponding to a 3-qubit QFT operator. Note that we can define any underlying unitary the same way.

The first step in our workflow decomposes our workflow to be more amenable to our final FT circuit. We'll define our worfklow to leverage multi-qubit synthesis to output a CNOT + continuous rotation circuit.

In [1]:
import numpy as np
# Import the circuit primitives from basic BQSKit
from bqskit.ir import Circuit
from bqskit.ir.gates import HGate, CNOTGate, ConstantUnitaryGate
from bqskit.qis.unitary import UnitaryMatrix

# Build a circuit with 6 qubits
circ = Circuit(num_qudits=6)

# We can define arbitrary unitary gates, such as QFT gate or Toffolis
qft_3_unitary = np.zeros((8, 8), dtype=np.complex128)

# Fill in the QFT matrix
for j in range(8):
    for k in range(8):
        qft_3_unitary[j, k] = np.exp(2j * np.pi * j * k / 8) / np.sqrt(8)


toffoli_unitary = np.eye(8, dtype=np.complex128)
toffoli_unitary[6, 6] = 0
toffoli_unitary[7, 7] = 0
toffoli_unitary[6, 7] = 1
toffoli_unitary[7, 6] = 1

qft_3_gate = ConstantUnitaryGate(
    utry=qft_3_unitary,
)
toffoli_gate = ConstantUnitaryGate(
    utry=toffoli_unitary,
)

# We can also do random matrices
random_un_gate_1 = ConstantUnitaryGate(UnitaryMatrix.random(2))
random_un_gate_2 = ConstantUnitaryGate(UnitaryMatrix.random(2))


# Add gates to the circuit
circ.append_gate(HGate(), [0])
circ.append_gate(CNOTGate(), [0, 1])
circ.append_gate(qft_3_gate, [2, 3, 4])
circ.append_gate(random_un_gate_1, [2, 5])

circ.append_gate(HGate(), [2])
circ.append_gate(CNOTGate(), [2, 5])
circ.append_gate(random_un_gate_2, [1,2])
circ.append_gate(toffoli_gate, [0, 1, 2])

print(circ.gate_counts)

{ConstantUnitaryGate: 1, ConstantUnitaryGate: 1, ConstantUnitaryGate: 1, CNOTGate: 2, ConstantUnitaryGate: 1, HGate: 2}


We can use Numerical Synthesis to decompose these arbitrary unitaries to a given gate set.

For our initial pass, we need to decompose into the Clifford gate set with continous single-qubit
rotation gates (U3 gates).

In [2]:
# Define the Multi-qudit retargeting pass
from bqskit.ir.gates.parameterized.u3 import U3Gate
from bqskit.passes.control.foreach import ForEachBlockPass
from bqskit.passes.partitioning.scan import ScanPartitioner
from bqskit.passes.synthesis import QSearchSynthesisPass
from bqskit.passes import UnfoldPass
from bqskit.compiler.gateset import GateSet


# We want to retarget this circuit to be in a Clifford + continuous rotation gate set
gate_set = GateSet({
    CNOTGate(),
    U3Gate(),
})

# We can use QSearch to perform a circuit search
qsearch_pass = QSearchSynthesisPass(
    layer_generator=gate_set.build_layer_generator()
)


multi_qudit_workflow = [
    ScanPartitioner(3), # Partition into 3-qubit blocks
    ForEachBlockPass(
        [qsearch_pass]
    ),
    UnfoldPass(), # Unfold the circuit back into a single circuit
]

In [3]:
# # Compile the circuit with the workflow
from bqskit.compiler.compiler import Compiler

compiler = Compiler(num_workers = 4)
# compiled_circ = compiler.compile(circ, multi_qudit_workflow)

# print(compiled_circ.gate_counts)


Decomposing to a Discrete Gate Set
----------
One of the most significant challenges in Fault-Tolerant compilation is the switch from continous rotations (e.g $Rs$, $U3$ gates) to a set of discrete rotations that can be performed by the underlying error correction code.

By far the most common gate set studied in the literature is the Clifford + $T$ gate set. While exact implementations vary signficantly between architectures, the most significant cost comes from the realization of the $T$ gate. These operations are done via magic state injection, which requires the cultivation/distillation of a resource T state, that then gets injected into the full circuit.

BQSKit-FT uses 3 distinct techniques to minimize these expensive T gates:

1) Multi-Qubit Synthesis (Dynamic Phase Fixing)
2) Direct U3 Synthesis
3) ZXZXZ Decomposition with Gridsynth


Dyadic Phase Fixing
----------
We leverage the technique described in [2] to numerically minimize the number of $T$ gates that comes from continous angle decomposition. At a high level, we perform a greedy search to "fix" as many continuous $Rz$ angles to multiples of $\frac{\pi}{4}$ which can be implemented in a single $T$ gate.

As demonstrated in the paper, this technique can save a signficant percentage (up to 70%) of $T$ gates when compared to naive compilation. We discuss later how we can extend this technique when we target an architecture with many ancilla.


In [4]:
# We will run this on the H2 molecule
h2_circ = Circuit.from_file('H_2.qasm')
print(h2_circ.gate_counts)

{CNOTGate: 33, U3Gate: 42}


The first step in this process to break down generic single-qubit unitaries as a sequence of Clifford gates and $R_Z$ gates:

$$
    U = R_Z(\theta) \sqrt{X} R_Z(\phi) \sqrt{X} R_Z(\lambda)
$$

In [5]:
# First we need to convert each U3 gate to 3 Rz gates
from bqskit.passes.rules import ZXZXZDecomposition
from bqskit.ir import Operation

zxzxz_decomposition = ZXZXZDecomposition()

# Make sure to run this pass on all single qudit gates
def single_qudit(op: Operation) -> bool:
    return op.num_qudits == 1

zxzxz = ForEachBlockPass(
    [ZXZXZDecomposition()], collection_filter=single_qudit,
)

h2_zxzxz_circ = compiler.compile(h2_circ, [zxzxz, UnfoldPass()])

print(h2_zxzxz_circ.gate_counts)


{RZGate: 126, CNOTGate: 33, SqrtXGate: 84}


We now can numerically collapse some $R_Z$ gates to dyadic angles (angles of the form $\frac{2\pi}{2^k}$).

These angles are particularly nice for FT computing, since they can be efficiently implemented with a phase addition circuit (we will talk about this more later).

However, for our tutorial (and in general if ancilla are not available), we will focus only on angles where $k=3$. This represents the set of gates that can be implemented by the $Z$, $S$, and $T$ gate set directly.

In [6]:
# Now run Dyadic Phase Fixing
from bqskit.ft.ftpasses.dpf import DPFPass
from bqskit.ft.ftpasses.convert_frz import ConvertFractionalRZPass

# Generate DyadicPhaseFixingPass
dpf_workflow = [
    DPFPass(k = 3, success_threshold=1e-3), # k is the maximum register size. k=3 is equal to T gates
    ConvertFractionalRZPass(max_k = 3) # Convert all Dyadic RZ gates to T gates
]

dpf_workflow = [
    ScanPartitioner(3), # This pass works much better on 4-qubit blocks
    ForEachBlockPass(
        dpf_workflow
    ),
    UnfoldPass()
]

In [7]:
# Compile the circuit with the workflow
# Note that we do not have to partition here because the circuit is 4 qubits
h2_dpf_circ = compiler.compile(h2_zxzxz_circ, dpf_workflow)

print(h2_dpf_circ.gate_counts)

{ZGate: 45, SGate: 48, RZGate: 44, TGate: 12, CNOTGate: 33, SqrtXGate: 84}


We were able to remove 80 Rz gates numerically!

In [8]:
# Let's confirm that this circuit is "close" to the original circuit
final_un = h2_dpf_circ.get_unitary()

orig_un = h2_circ.get_unitary()

print(final_un.get_distance_from(orig_un))

1.4901161193847656e-08


Direct U3 Synthesis
----------
After performing Dyadic Phase Fixing, we are left with our fixed $T$ gates, with the remaining set of continuous rotations.

To decompose this set of continuous rotations, we switch from numerical synthesis to an algebraic approach to gate decomposition. Following the technique described in [3], we can directly decompose *any* single qubit unitary to Clifford + T directly. While this is an expensive 8-dimensional lattce search, we see in practice that this search can be performed efficiently up to precisions of 10e-8, saving an additional 66% of $T$ gates when compared to gridsynth.

In [9]:
# Let's take the output of the DPF Synthesis and convert the Rzs back to U3s

# First, let's recombine the Rz gates back to U3 gates
from cyclopass import CombineRotationsPass

workflow = [
    CombineRotationsPass()
]


In [10]:
h2_dpf_u3_circ = compiler.compile(h2_dpf_circ, workflow)
print(h2_dpf_u3_circ.gate_counts)

{ZGate: 45, SGate: 48, U3Gate: 34, TGate: 12, CNOTGate: 33, SqrtXGate: 53}


In [11]:
# Let's use the U3 Synthesis pass
from cyclopass import U3ToTPass
u3_to_t_pass = U3ToTPass(epsilon=1e-3)

u3_to_t_workflow = [u3_to_t_pass]

In [12]:
from bqskit.ir.gates import TGate, TdgGate
# Let's compile both 
full_decomp_circ = compiler.compile(h2_dpf_u3_circ, u3_to_t_workflow)

print("Final Gate Counts: ", full_decomp_circ.gate_counts)
print("Final T Counts:", full_decomp_circ.count(TGate()) + full_decomp_circ.count(TdgGate()))

full_decomp_un = full_decomp_circ.get_unitary()
print("Distance: ", full_decomp_un.get_distance_from(orig_un))


Final Gate Counts:  {ZGate: 48, SdgGate: 101, TGate: 620, XGate: 2, SqrtXGate: 53, SGate: 216, U3Gate: 20, YGate: 1, TdgGate: 62, CNOTGate: 33, HGate: 892}
Final T Counts: 682
Distance:  2.7848835130517656e-05


ZXZXZ Decomposition and GridSynth
----------
Finally, if there are remaining angles that need to be decomposed after the first 2 techniques, we default to the Ross-Selinger "gridsynth" algorithm described in [4]. This algorithm produces optimal length sequences of Clifford+$T$ gates approximating $R_Z(\theta)$ rotations. 

As mentioned above, each general single qubit unitaries can be broken down into 3 $R_Z(\theta)$ gates. This means, on average, that this decomposition is 3x more expensive that direct $U3$ synthesis, but still represents the current state-of-the-art algorithm for high precision gate synthesis.

In [13]:
# Instead of using cyclosynth, we can default to ZXZXZ decomp + Rz synthesis
h2_zxzxz_circ = compiler.compile(h2_circ, [zxzxz, UnfoldPass()])

print(h2_zxzxz_circ.gate_counts)


{RZGate: 126, CNOTGate: 33, SqrtXGate: 84}


In [17]:
# Now compile each RZ Gate with Gridsynth
from bqskit.ft.rules.isolate_rz import IsolateRZGatePass
from bqskit.ft.ftpasses.gridsynth import GridSynthPass
rz_workflow = [
        IsolateRZGatePass(),
        ForEachBlockPass([GridSynthPass(precision=5)]),
        UnfoldPass(),
    ]

In [18]:
# Now run the workflow on the ZXZXZ circuit
h2_gridsynth_circ = compiler.compile(h2_zxzxz_circ, rz_workflow)

In [19]:
print("Default (ZXZXZ + Gridsynth) Gate Counts: ", h2_gridsynth_circ.gate_counts)
print("Default (ZXZXZ + Gridsynth) T Counts: ", h2_gridsynth_circ.count(TGate()) + h2_gridsynth_circ.count(TdgGate()))

gridsynth_un = h2_gridsynth_circ.get_unitary()
print("Distance: ", gridsynth_un.get_distance_from(orig_un))

Default (ZXZXZ + Gridsynth) Gate Counts:  {CNOTGate: 33, SGate: 1883, HGate: 3334, TGate: 3286, XGate: 41, SqrtXGate: 84}
Default (ZXZXZ + Gridsynth) T Counts:  3286
Distance:  3.9007363689619966e-05


Algorithmic Error
-----------------

To this point, we have avoided talking about a fundamental component of fault-tolerant compilation: approximation error. Approximation error is necessary in fault-tolerant compilation as we try to express continous rotation gates from different algorithmic domains to a sequence of Clifford+$T$ gates.

We have set up our BQSKit workflows to bound the total approximation across the entire circuit, regardless of circuit size. We can do this via *circuit partitioning*

As per [4], we can bound the error across the entire circuit as the *sum* of the error across each partition.

Therefore, we can define the following partitioning scheme to ensure the approximation error across the entire algorithm.

In [ ]:
def algorithmic_error_workflow(circ: Circuit, 
                               block_size: int,
                               algorithmic_error: float,
                               compiler: Compiler) -> list:
    # First Partition circuit
    part_circ = compiler.compile(circ, [ScanPartitioner(block_size)])

    # Now allocate error
    gg_prec = algorithmic_error / circ.num_params
    gg_prec_int = int(np.ceil(-np.log10(gg_prec)))
    workflow = [
        ScanPartitioner(block_size),
        ForEachBlockPass(
            [
                # Divide error for each partition
                zxzxz,
                DPFPass(k = 3, success_threshold=algorithmic_error / part_circ.num_operations),
                ConvertFractionalRZPass(max_k = 3),
                CombineRotationsPass(),
                U3ToTPass(epsilon=algorithmic_error / part_circ.num_operations),
                # Default if Cyclosynth fails
                zxzxz,
                UnfoldPass(),
                IsolateRZGatePass(),
                # Pass in precision as algorithmic_error / part_circ.num_operations
                # Rounded to near 10 ** -(x)
                ForEachBlockPass([GridSynthPass(precision=gg_prec_int)]),
                UnfoldPass(),
            ]
        ),
        UnfoldPass()
    ] 

    return workflow

In [19]:
# Let us run the full workflow at different error levels on a large circuit
lgt_circ = Circuit.from_file('lgt_11.qasm')

print("Original Gate Counts:", lgt_circ.gate_counts)

compiler.close()
compiler = Compiler(num_workers = 4)

workflow_1e_2 = algorithmic_error_workflow(lgt_circ, 3, 1e-2, compiler)
workflow_1e_4 = algorithmic_error_workflow(lgt_circ, 3, 1e-4, compiler)

lgt_circ_1e_2 = compiler.compile(lgt_circ, workflow_1e_2)
lgt_circ_1e_4 = compiler.compile(lgt_circ, workflow_1e_4)

print("Gate Counts (1e-2):", lgt_circ_1e_2.gate_counts)
print("Gate Counts (1e-4):", lgt_circ_1e_4.gate_counts)

Original Gate Counts: {CNOTGate: 76, U3Gate: 72}


Compiler interrupted.


KeyboardInterrupt: 

Pauli Product Measurements
--------------------------

In many cases for FT compilation, we want our final output to be a sequence of Pauli Product Measurements.

To do this, use a Clifford Tableau to conjugate gates and output a sequence of 
Pauli Product Measurements and an associated rotation angle.

In [ ]:
from ppms.ppm_transpile import PPMTranspilePass

In [ ]:
from bqskit.compiler import Compiler

# Pass in Clifford + T H_2 circ to PPMTranspilePass

compiler = Compiler(num_workers=1)
out_circ = compiler.compile(full_decomp_circ, workflow=[PPMTranspilePass()])

In [ ]:
from ppms import PPMPlaceholder
for op in out_circ.operations():
    # Format as X1Z2 with base_qubitnum
    assert isinstance(op.gate, PPMPlaceholder)
    name = ""
    if op.params[1] == 1:
        name += "-"
    for base, qubit in zip (op.gate.bases, op.location):
        name += f"{base}{qubit}"

    if op.params[0] == 0.25:
        name += "(pi/4)"
    print(name)

-Y0(pi/4)
-Y1(pi/4)
-Y3(pi/4)
Z1(pi/4)
-Y1(pi/4)
X1(pi/4)
Z1(pi/4)
X1(pi/4)
Z1(pi/4)
-Y1(pi/4)
Z1(pi/4)
X1(pi/4)
-Y1(pi/4)
Z1(pi/4)
X1(pi/4)
Z1(pi/4)
X1(pi/4)
-Y1(pi/4)
X1(pi/4)
-Y1(pi/4)
X1(pi/4)
Z1(pi/4)
X1(pi/4)
Z1(pi/4)
X1(pi/4)
-Y1(pi/4)
Z1(pi/4)
-Y1(pi/4)
Z1(pi/4)
X1(pi/4)
-Y1(pi/4)
X1(pi/4)
Z1(pi/4)
X1(pi/4)
Z1(pi/4)
X1(pi/4)
Z1(pi/4)
-Y1(pi/4)
X1(pi/4)
-Y1(pi/4)
Z1(pi/4)
-Y1(pi/4)
Z1(pi/4)
X1(pi/4)
Z1(pi/4)
-Y1(pi/4)
Z1(pi/4)
X1(pi/4)
Z1(pi/4)
-Y1(pi/4)
Z1(pi/4)
X1(pi/4)
-Y1(pi/4)
-Y0Y1(pi/4)
Z1(pi/4)
-Y0Y1(pi/4)
Y0X1(pi/4)
-Y0Y1(pi/4)
Y0X1(pi/4)
Z1(pi/4)
Y0X1(pi/4)
-Y0Y1(pi/4)
Y0X1(pi/4)
Z1(pi/4)
-Y0Y1(pi/4)
Y0X1(pi/4)
Z1(pi/4)
Y0X1(pi/4)
Z1(pi/4)
Y0X1(pi/4)
Z1(pi/4)
-Y0Y1(pi/4)
Y0X1(pi/4)
Z1(pi/4)
Y0X1(pi/4)
Z1(pi/4)
-Y0Y1(pi/4)
Y0X1(pi/4)
-Y0Y1(pi/4)
Z1(pi/4)
-Y0Y1(pi/4)
Z1(pi/4)
-Y0Y1(pi/4)
Y0X1(pi/4)
-Y0Y1(pi/4)
Y0X1(pi/4)
Z1(pi/4)
Y0X1(pi/4)
Z1(pi/4)
Y0X1(pi/4)
Z1(pi/4)
Y0X1(pi/4)
Z1(pi/4)
Y0X1(pi/4)
-Y0Y1(pi/4)
Y0X1(pi/4)
Z1(pi/4)
-Y0Y1(pi/4)
Z1(pi/4)
Y0X1(pi/4)
-Y0Y1(pi